# HW3 · 02 — Baseline: TF-IDF + linear models
Word and char n-gram TF-IDF, Logistic Regression and Linear SVM, a small grid, per-class metrics and a confusion matrix.
Vectorizers are fit on `train_sub` ONLY (no leakage from val/test).

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay

SEED = 42
MAX_CHARS = 5000  # TF-IDF sees the first 5000 characters (speed/memory); most texts are shorter

DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
TEXT_COL, LABEL_COL = "text", "label"
LANG_COL = next((c for c in ["language", "lang"] if c in train.columns), None)

In [ ]:
# Same split as notebook 01
def make_split(train, label_col="label", lang_col=None, n_val=10_000, n_train=40_000, seed=42):
    strat = train[label_col].astype(str) + ("_" + train[lang_col].astype(str) if lang_col else "")
    rest, val = train_test_split(train, test_size=n_val, stratify=strat, random_state=seed)
    strat_rest = strat.loc[rest.index]
    train_sub, _ = train_test_split(rest, train_size=n_train, stratify=strat_rest, random_state=seed)
    return train_sub.reset_index(drop=True), val.reset_index(drop=True)

train_sub, val = make_split(train, LABEL_COL, LANG_COL)
print(train_sub.shape, val.shape, "first val ids:", val["id"].head(3).tolist())

X_tr = train_sub[TEXT_COL].str[:MAX_CHARS]
X_va = val[TEXT_COL].str[:MAX_CHARS]
X_te = test[TEXT_COL].str[:MAX_CHARS]
y_tr, y_va = train_sub[LABEL_COL].values, val[LABEL_COL].values
CLASSES = np.array(sorted(train[LABEL_COL].unique()))
print(CLASSES)

## Feature sets (fit on train_sub only)

In [ ]:
t0 = time.time()
word_vec = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=3, max_features=200_000,
                           sublinear_tf=True, lowercase=True)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=5, max_features=300_000,
                           sublinear_tf=True, lowercase=False)  # keep case: capitalisation is a style signal
Xw_tr = word_vec.fit_transform(X_tr); Xw_va = word_vec.transform(X_va)
Xc_tr = char_vec.fit_transform(X_tr); Xc_va = char_vec.transform(X_va)
features = {
    "word 1-2": (Xw_tr, Xw_va),
    "char_wb 2-5": (Xc_tr, Xc_va),
    "word+char": (hstack([Xw_tr, Xc_tr]).tocsr(), hstack([Xw_va, Xc_va]).tocsr()),
}
print(f"vectorizing took {time.time() - t0:.0f}s; shapes:", {k: v[0].shape for k, v in features.items()})

## Small grid: feature set × model × C  (selection by macro-F1 on the fixed val split)

In [ ]:
models = {
    "LogReg C=1": lambda: LogisticRegression(C=1, max_iter=2000),
    "LogReg C=4": lambda: LogisticRegression(C=4, max_iter=2000),
    "LinearSVC C=0.1": lambda: LinearSVC(C=0.1),
    "LinearSVC C=0.5": lambda: LinearSVC(C=0.5),
}
rows, fitted = [], {}
for fname, (Xtr, Xva) in features.items():
    for mname, make in models.items():
        t0 = time.time()
        clf = make().fit(Xtr, y_tr)
        pred = clf.predict(Xva)
        rows.append({"features": fname, "model": mname, "macro_f1": f1_score(y_va, pred, average="macro"),
                     **dict(zip([f"f1_{c}" for c in CLASSES], f1_score(y_va, pred, average=None, labels=CLASSES))),
                     "sec": round(time.time() - t0)})
        fitted[(fname, mname)] = clf
        print(rows[-1])
grid = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
grid.round(4).to_csv("baseline_grid.csv", index=False)
grid.round(4)

## Extra check: class_weight="balanced" for the best configuration

In [ ]:
best = grid.iloc[0]
Xtr, Xva = features[best["features"]]
clf_bal = models[best["model"]]().set_params(class_weight="balanced").fit(Xtr, y_tr)
print("best:", best["features"], best["model"], round(best["macro_f1"], 4),
      "| balanced:", round(f1_score(y_va, clf_bal.predict(Xva), average="macro"), 4))

## Per-class report and confusion matrix (best configuration)

In [ ]:
best_clf = fitted[(best["features"], best["model"])]
pred = best_clf.predict(Xva)
print(classification_report(y_va, pred, digits=4))
if LANG_COL:
    for lang, g in val.groupby(LANG_COL):
        m = (val[LANG_COL] == lang).values
        print(f"{lang}: macro-F1 = {f1_score(y_va[m], pred[m], average='macro'):.4f}  (n={m.sum()})")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(y_va, pred, labels=CLASSES, ax=ax[0], colorbar=False)
ConfusionMatrixDisplay.from_predictions(y_va, pred, labels=CLASSES, normalize="true", values_format=".2f", ax=ax[1], colorbar=False)
ax[0].set_title("counts"); ax[1].set_title("row-normalised (recall)")
plt.tight_layout(); plt.savefig("baseline_confusion.png", dpi=120); plt.show()

## Submission + saved probabilities (Logistic Regression on word+char, used later for the ensemble)

In [ ]:
def transform(fname, texts):
    if fname == "word 1-2": return word_vec.transform(texts)
    if fname == "char_wb 2-5": return char_vec.transform(texts)
    return hstack([word_vec.transform(texts), char_vec.transform(texts)]).tocsr()

test_pred = best_clf.predict(transform(best["features"], X_te))
sub = pd.DataFrame({"id": test["id"], "label": test_pred})
sub = sample_sub[["id"]].merge(sub, on="id", how="left")
assert sub["label"].notna().all() and len(sub) == len(sample_sub)
sub.to_csv("submission_baseline.csv", index=False)
print(sub["label"].value_counts(normalize=True).round(3))

# probabilities for the ensemble notebook
lr_key = max([k for k in fitted if k[1].startswith("LogReg")], key=lambda k: grid.set_index(["features", "model"]).loc[k, "macro_f1"])
lr = fitted[lr_key]
print("LogReg used for probabilities:", lr_key)
pv = lr.predict_proba(transform(lr_key[0], X_va)); pt = lr.predict_proba(transform(lr_key[0], X_te))
pd.DataFrame(pv, columns=lr.classes_).assign(id=val["id"].values).to_csv("baseline_val_proba.csv", index=False)
pd.DataFrame(pt, columns=lr.classes_).assign(id=test["id"].values).to_csv("baseline_test_proba.csv", index=False)

## Top n-grams per class (quick global explanation, reused for the xAI part)

In [ ]:
names = np.array(word_vec.get_feature_names_out().tolist() + char_vec.get_feature_names_out().tolist()) \
    if lr_key[0] == "word+char" else (word_vec if lr_key[0] == "word 1-2" else char_vec).get_feature_names_out()
for i, c in enumerate(lr.classes_):
    top = np.argsort(lr.coef_[i])[-20:][::-1]
    print(f"{c:7s}:", " | ".join(repr(n) for n in names[top]))